# Session 2 — Build an Agentic RAG System

In this session you will build, step by step, an agent that answers
questions about NVIDIA's financial results using the company's own filings.

The pipeline has four stages:

1. **Chunk** 
2. **Vectorize** 
3. **Search** 
4. **Assemble an agent** 

The corpus is in `data/02_processed/`: NVIDIA's 10-K filings for fiscal 2025 and 2026,
trimmed to their substantive pages, plus the transcripts of the two matching earnings
calls.

Before starting, take a look at the documents themselves in `data/01_raw/` and
`data/02_processed/` — that is the content everything here relies on.

## Setup

Helpers are given to you in `src/utils_build.py` — open it and have a look.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.append("../src")

from utils_build import build_chunk_records, load_document, save_chunks

PROCESSED_DIR = Path("../data/02_processed")
CHUNKS_DIR = Path("../data/03_chunks")
ALL_CHUNKS_PATH = CHUNKS_DIR / "all_chunks.json"

sorted(p.name for p in PROCESSED_DIR.iterdir() if p.suffix in {".pdf", ".txt"})

## Exercise 1 — Chunking the documents

The first step of a RAG system is cutting the corpus into small pieces: how much a model
can read at once is limited, and the longer the context, the more it loses track of what
matters. Two parameters set the trade-off:

- **`chunk_size`** — maximum characters in a chunk. Too large wastes tokens and buries the
  answer among irrelevant text; too small truncates the information.
- **`overlap`** — characters shared by consecutive chunks. Without it, a sentence cut by a
  boundary is whole in neither.

### TO_DO_BENJAMIN 
[schema to insert here]_

**Objective.** Implement the chunking logic: splitting a document's text into overlapping
pieces.

**Your task.** Complete `chunk_string` below, respecting its signature and the intent
documented in its docstring.

**Hint.** Use [`CharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/CharacterTextSplitter)
from `langchain_text_splitters`. 

In [ ]:
def chunk_string(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into overlapping, non-blank chunks.

    Parameters
    ----------
    text : str
        The text to split.
    chunk_size : int
        Maximum number of characters in a chunk.
    overlap : int
        Number of characters each chunk shares with the previous one.

    Returns
    -------
    list[str]
        The chunks, in order. Blank chunks are dropped.
    """
    # YOUR CODE HERE
    raise NotImplementedError

### Check your implementation

Small enough to check by hand: with `chunk_size=10` and `overlap=3`, consecutive chunks
start 7 characters apart.

In [ ]:
chunks = chunk_string("abcde, fghij. klmn\n opqrs!", chunk_size=10, overlap=3)

assert chunks == ["abcde, fgh", "fghij. klm", "klmn\n opqr", "pqrs!"], chunks
print("OK —", len(chunks), "chunks")

### Chunk every document

Four steps per document, one file each in `data/03_chunks/`:

- `load_document(path)` — returns the document's full text and the fiscal year it covers.
- your `chunk_string` — cuts that text into overlapping pieces.
- `build_chunk_records(text_chunks, document_name, year)` — gives each chunk an `id` and
  records which document and year it came from.
- `save_chunks(records, path)` — writes the records to a JSON file.

In [ ]:
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

for document_path in sorted(PROCESSED_DIR.iterdir()):
    if document_path.suffix not in {".pdf", ".txt"}:
        continue  # skip anything that is not one of our documents

    text, year = load_document(str(document_path))
    text_chunks = chunk_string(text, chunk_size=1000, overlap=200)
    records = build_chunk_records(text_chunks, document_path.name, year)

    save_chunks(records, str(CHUNKS_DIR / f"{document_path.stem}.json"))
    print(f"{document_path.name}: {len(records)} chunks")

### Gather them into one file

Searching should look through one list, not four. This reads every per-document file back
and concatenates them into a single `all_chunks.json`, which is what the next exercises
will work from.

In [ ]:
all_chunks = []

for document_chunks_path in sorted(CHUNKS_DIR.glob("*.json")):
    if document_chunks_path != ALL_CHUNKS_PATH:  # do not read the gathered file into itself
        all_chunks.extend(json.loads(document_chunks_path.read_text()))

save_chunks(all_chunks, str(ALL_CHUNKS_PATH))
print(f"{len(all_chunks)} chunks in {ALL_CHUNKS_PATH}")